# Python 101
## Part VIII.

---

## Web Scraping - Part II.

### Act I: Let's scrape!

But first, import the necessary libraries.

In [ ]:
import datetime
import requests
from bs4 import BeautifulSoup

BASE_URI = './data/'

#### Exercise 1. Collect the articles about Soros from portfolio.hu

This requires searching the site.
On the upper-left corner, there is a search icon. Use it, and observe the resulting url:

`https://portfolio.hu/kereses?q=Soros&df=1999-02-10&dt=2025-10-27&page=1`

It has multiple parts:
- `https://` - protocol
- `portfolio.hu` - base url
- `/kereses` - sub url
- `?q=Soros&df=1999-02-10&dt=2025-10-27&page=1` - query

Let's investigate the query part a little more!  
Every query starts with a __`?`__ character followed by one or more key-value pairs. The key-value pairs are separated with the __`&`__ character. Based on this information, we can extract the query parameters:
- `df` - `date from`
- `dt` - `date to`
- `q` - stands for query (the word we are looking for)
- `page` - page number

Use these values to construct our own request:

In [ ]:
today = datetime.date.today().isoformat()
base_url = 'https://portfolio.hu'
sub_url = '/kereses'
query = {
    'q': 'Soros',
    'df': '1999-02-09',
    'dt': today,
    'page': 1
}

We can use the requests library to send the query:

In [ ]:
response = requests.get(url=base_url+sub_url, params=query) # forms sent with POST need requests.post(url, data=...) instead
response

Using the response, extract the urls from the articles! Pay attention, you may find `<article>` tags in weird places that you do not want to include.

You can see that only 20 results showed up. We can customize our query to cover a shorter period of time by replacing __`df`__ and __`dt`__ parameters with a formattable string: __`'{year}-{month:0>2}-{day:0>2}'`__. This string can be formatted by providing the required parameters:
- year
- month
- day

like this:

In [ ]:
'{year}-{month:0>2}-{day:0>2}'.format(year=2016, month=1, day=1)

There is a useful library called __`datetime`__. You can use it to generate dates automatically.

In [ ]:
date = datetime.date(1999, 1, 1)
one_day = datetime.timedelta(days=1)
day_after_date = date + one_day
day_before_date = date - one_day
today = datetime.date.today()

print(day_before_date)
print(date)
print(day_after_date)
print(today)

print(today.year, today.month, today.day)
print(today.isoformat())   # the same 'YYYY-MM-DD' string as the format above

Create a loop which iterates through every day from 1999-01-01 till today and executes the same procedure you created previously. (Pro tip: create a function!) Observe the number of results!

**Be gentle with the site:** one request per day since 1999 is almost 10 000 requests. Test your loop on a short range first (e.g. the last week), and wait a little between two requests with `time.sleep(1)` (`import time`) - a whole classroom hammering the site at once looks like an attack.

---

### Act II: Disguise yourself!

![Disguise](pics/batman-superman-disguise.gif)

Let's pretend to be a browser instead of a script:

In [ ]:
USER_AGENTS = [
    # Windows PC using a Chrome browser
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36',
    # Windows PC using an Edge browser
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36 Edg/140.0.0.0',
    # Mac using a Safari browser
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.6 Safari/605.1.15',
    # Linux PC using a Firefox browser
    'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:143.0) Gecko/20100101 Firefox/143.0',
    # Chromebook using a Chrome browser
    'Mozilla/5.0 (X11; CrOS x86_64 14541.0.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36',
]

You can find more user agents [here](https://deviceatlas.com/blog/list-of-user-agent-strings). Use recent ones: some sites refuse browsers that are years out of date.

Let's write a wrapper function to handle the user-agent string.

In [ ]:
import random
def get_header(agents):
    return {'User-agent': random.choice(agents)}

#### Get the main articles from telex.hu
Write a function that extracts the current main articles! It should contain:
- the title
- the article text
- the url
- every picture from the article

Hint(s):
- open telex.hu in your browser, right-click a headline and pick **Inspect** to see its tags and classes
- the links on the front page are relative (`/belfold/...`) - put `https://telex.hu` in front of them
- the same article can be linked more than once (picture + headline)

In [ ]:
url = 'https://telex.hu'
telex_response = requests.get(url, headers=get_header(USER_AGENTS))

# raise an error in case the download was unsuccessful
telex_response.raise_for_status()

#### Exercise 2. What is on sale on Steam right now?

Steam publishes its current discounts as a plain, server-rendered page - no javascript needed:

`https://store.steampowered.com/search/?specials=1`

Every game on it sits in an `<a>` tag with the `search_result_row` class, and inside that you will find:

| what | selector |
|---|---|
| title | `.title` |
| discount | `.discount_pct` |
| original price | `.discount_original_price` |
| current price | `.discount_final_price` |
| review summary | `.search_review_summary` |
| release date | `.search_released` |

Collect the **title, discount, original price and current price** of every game on the first page.

*Don't forget the disguise from Act II - send a `User-agent` header.*

Extra questions:
- How much would it cost to buy **everything** on the page at the sale price?
- How much would the same shopping list have cost at full price - and how much do you save in total?
- Which game has the **deepest discount**, and which one saves you the **most money**?
- The page only shows the first batch. Add `&page=2` to the url and collect a few pages.
  Watch out for a quirk: with **no** `page` parameter Steam returns 50 games, but as soon
  as you pass `page=1` it returns 25. Check the number of rows you actually got instead of
  assuming - that habit will save you a lot of debugging.

*Hint: the prices come back as strings like `'11,99€'`. You will have to strip the currency symbol and turn the comma into a dot before you can add them up.*

#### Exercise 3. Functionize!

##### a. Create a function which looks up the current price of one game

Steam's search accepts a `term` parameter: `https://store.steampowered.com/search/?term=half-life+2`.
Take the first result and return its price.

In [ ]:
def check_price(game):
    pass

##### b. Create a function which collects the games on sale (with a `pages` argument)

In [ ]:
def get_specials(pages=1):
    pass

##### c. Write a function with the same functionality as the 2nd exercise!

In [ ]:
def main():
    pass

---

### Intermission: Creating a standalone script

Create a new text file with `.py` extension **next to this notebook** (otherwise `import` will not find it). You can choose the filename - below we call it `myscript.py`.

Copy into it:
- the imports,
- the global variables (`base_url`, `sub_url`, `query`, `USER_AGENTS`),
- the functions from Exercise 3, and every helper they use (like `get_header`),

and add these two lines at the end of the file:

```python
if __name__ == '__main__':
    main()
```

Save it, and now you can execute this script from the Anaconda Prompt / Terminal by invoking:

```bash
python myscript.py
```

The `if __name__ == '__main__':` part runs only when you execute the file - not when you import it.

You can even:
- import your newly created script (if you change the file later, restart the kernel before importing it again):

In [ ]:
import myscript # use your filename

- get its contents

In [ ]:
dir(myscript)

- print its variables

In [ ]:
print(myscript.base_url, myscript.sub_url, myscript.query)

- use its functions

In [ ]:
hl2 = myscript.check_price('half-life 2')
print(hl2)

In [ ]:
myscript.get_specials(pages=2)

---

## Let's do some...

<img align="left" width=150 src="pics/magic.gif">
<br style="clear:left;"/>

### Act III: Cool library of the week: Tkinter
#### Create graphical user interfaces!
All you have to do is:
- Import it

In [ ]:
import random
import tkinter

- Create a class:
    - with window layout
    - with function bindings 

In [ ]:
class Dice(tkinter.Tk):

    def __init__(self):
        # set up the main window
        super().__init__()
        self.initialize()

    def initialize(self):
        self.grid()

        # add label
        self.labelVariable = tkinter.StringVar()
        label = tkinter.Label(self, textvariable=self.labelVariable)
        label.grid(column=0, row=0, sticky='EW')
        self.labelVariable.set(0)

        # add button
        button = tkinter.Button(self, text='Throw!', command=self.throw)
        button.grid(column=1, row=0)

    def throw(self):
        self.labelVariable.set(random.randint(1, 6))


- Initiate and use it

**Careful:** `mainloop()` blocks the notebook until you close the window - the
cell keeps its `[*]` and the kernel looks frozen. That is normal. It is
commented out below so that *Run All* works; uncomment it when you want to play
with the dice.

In [ ]:
# app = Dice()
# app.title('Throw a dice!')
# app.mainloop()

An easy to follow tutorial can be found [here](https://tkdocs.com/tutorial/), and the reference in the [python documentation](https://docs.python.org/3/library/tkinter.html).

---

## Final Act: It's your turn - write the missing code snippets!

Write a script called `articles.py`, next to this notebook, in which you create a function called `get_article_tags` and a global variable called `BASE_URL` (telex.hu's base url).


`get_article_tags(telex_article_suburl)`:
- Arguments: telex_article_suburl
- Output: the list of the article's topic tags (the labels listed on the article page - not HTML tags!), as strings
- Workflow:
    1. Combine BASE_URL and the article sub-URL to form the full URL.
    1. Send a disguised HTTP request (use a User-Agent header).
    1. Parse the page with BeautifulSoup.
    1. Find the topic tags (they are `<a>` elements - inspect the page to find their class) and return their texts as a list.

Don't forget about the disguise!

In [ ]:
# test the script
import articles

In [ ]:
# paste the part of any telex article's url after 'telex.hu' here, e.g. '/belfold/2026/...'
# (an empty string gives the front page, which has no topic tags)
related_tags = articles.get_article_tags('')

In [ ]:
for tag in related_tags:
    print(tag)